In [1]:
pip install selenium beautifulsoup4 pandas webdriver-manager

  Using cached trio_websocket-0.12.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
  Using cached wsproto-1.3.2-py3-none-any.whl.metadata (5.2 kB)
  Using cached PySocks-1.7.1-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.6 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.6 MB 1.1 MB/s eta 0:00:09
   -- ------------------------------------- 0.5/9.6 MB 1.1 MB/s eta 0:00:09
   --- ------------------------------------ 0.8/9.6 MB 1.0 MB/s eta 0:00:09
   --- ------------------------------------ 0.8/9.6 MB 1.0 MB/s eta 0:00:09
   ---- ----------------------------------- 1.0/9.6 MB 773.6 kB/s eta 0:00:12
   ---- ----------------------------------- 1.0/9.6 MB 773.6 kB/s eta 0:00:12
   ---- ----------------------------------- 1.0/9.6 MB 773.6

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dbt-snowflake 1.10.1 requires certifi<2025.4.26, but you have certifi 2026.4.22 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import time

In [ ]:
def get_crypto_data(total_pages=85):
    # --- Browser Setup ---
    options = webdriver.ChromeOptions()
    options.add_argument('--headless') 
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    
    all_rows_list = []

    try:
        for page_num in range(1, total_pages + 1):
            url = f"https://coinmarketcap.com/?page={page_num}"
            driver.get(url)
            print(f"Scraping Page {page_num}...")
            
            # --- Dynamic Scrolling (Lazy Loading Fix) ---
            # To Load all data, we scroll down multiple times with a delay
            for _ in range(10):
                driver.execute_script("window.scrollBy(0, 1000);")
                time.sleep(1) # Data load hone ka intezar

            # Poore page ki HTML BeautifulSoup ko dena
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            table = soup.find('table', class_='cmc-table')
            
            if not table:
                print(f"Table nahi mila page {page_num} par.")
                continue
                
            rows = table.find('tbody').find_all('tr')

            for index, row in enumerate(rows):
                # Condition: 1st page ki 1st row (CMC 20 Index) skip karni hai
                if page_num == 1 and index == 0:
                    continue
                
                cells = row.find_all('td')
                if len(cells) < 10:
                    continue

                try:
                    # 1. Rank
                    rank = cells[1].get_text(strip=True)

                    # 2. Name (Multiple tags check taake N/A na aaye)
                    name_cell = cells[2]
                    name_p = name_cell.find('p', class_='coin-item-name')
                    name = name_p.get_text(strip=True) if name_p else name_cell.get_text(" ", strip=True).split()[0]

                    # 3. Price
                    price = cells[3].get_text(strip=True)

                    # 4. 1h %
                    h1 = cells[4].get_text(strip=True)

                    # 5. 24h %
                    h24 = cells[5].get_text(strip=True)

                    # 6. 7d %
                    d7 = cells[6].get_text(strip=True)

                    # 7. Market Cap (Cleaning logic)
                    m_cap_raw = cells[7].get_text(strip=True)
                    # Kabhi kabhi do values hoti hain, hum pehli value ($ ke sath) uthayenge
                    m_cap = "$" + m_cap_raw.split('$')[1] if '$' in m_cap_raw else m_cap_raw

                    # 8. Volume(24h) (Updated for your issue)
                    vol_cell = cells[8]
                    # Volume ke liye 'font_weight_500' class ya normal 'p' ya 'a' tag check karna
                    vol_p = vol_cell.find('p', class_='font_weight_500') or vol_cell.find('p') or vol_cell.find('a')
                    
                    if vol_p:
                        volume = vol_p.get_text(strip=True)
                    else:
                        # Agar koi tag na mile to pure cell ka text le lo aur unnecessary part hata do
                        volume = vol_cell.get_text(strip=True).split(' ')[0]

                    # List mein data add karna
                    all_rows_list.append([rank, name, price, h1, h24, d7, m_cap, volume])

                except Exception:
                    continue # Agar kisi row mein error ho to skip karke agli row pe jao

    finally:
        driver.quit() # Browser band karna

    # --- DataFrame Creation ---
    columns = ['Rank', 'Name', 'Price', '1h%', '24h%', '7d%', 'Market Cap', 'Volume(24h)']
    df = pd.DataFrame(all_rows_list, columns=columns)
    return df

In [12]:
final_df = get_crypto_data(total_pages=2)

Scraping Page 1...
Scraping Page 2...


In [13]:
final_df

,Rank,Name,Price,1h%,24h%,7d%,Market Cap,Volume(24h)
0,1,Bitcoin,"$79,655.86",0.99%,1.24%,1.83%,$1.59T,"$45,094,955,140"
1,2,Ethereum,"$2,355.82",0.80%,1.35%,1.21%,$284.02B,"$22,402,210,819"
2,3,Tether,$0.9997,0.01%,0.01%,0.04%,$189.53B,"$138,793,308,342"
3,4,XRP,$1.40,0.37%,0.57%,0.47%,$86.63B,"$2,231,367,497"
4,5,BNB,$625.76,0.35%,0.99%,0.36%,$84.34B,"$2,107,281,764"
...,...,...,...,...,...,...,...,...
195,196,Safe,$0.1418,0.20%,2.35%,1.99%,$105.04M,"$3,376,243"
196,197,Gas,$1.61,0.16%,0.78%,3.02%,$104.97M,"$3,088,461"
197,198,Axelar,$0.08917,1.00%,21.69%,59.88%,$103.76M,"$78,482,269"
198,199,Irys,$0.04085,0.06%,5.58%,21.85%,$104.87M,"$8,167,782"


In [14]:
final_df.to_csv('coinmarketcap_data_selenium_bs4.csv', index=False)

In [15]:
print(final_df)

    Rank                   Name       Price    1h%    24h%     7d% Market Cap  \
0      1                Bitcoin  $79,655.86  0.99%   1.24%   1.83%     $1.59T   
1      2               Ethereum   $2,355.82  0.80%   1.35%   1.21%   $284.02B   
2      3                 Tether     $0.9997  0.01%   0.01%   0.04%   $189.53B   
3      4                    XRP       $1.40  0.37%   0.57%   0.47%    $86.63B   
4      5                    BNB     $625.76  0.35%   0.99%   0.36%    $84.34B   
..   ...                    ...         ...    ...     ...     ...        ...   
195  196                   Safe     $0.1418  0.20%   2.35%   1.99%   $105.04M   
196  197                    Gas       $1.61  0.16%   0.78%   3.02%   $104.97M   
197  198                 Axelar    $0.08917  1.00%  21.69%  59.88%   $103.76M   
198  199                   Irys    $0.04085  0.06%   5.58%  21.85%   $104.87M   
199  200  Official Melania Meme     $0.1033  0.32%   0.14%   6.60%   $103.35M   

          Volume(24h)  
0  